In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% ! important;}
div.cell.code_cell.rendered{width:100%}
div.input_prompt{padding:0px}
div.CodeMirror {font-family:Consolas ; font-size:12pt;}
div.text_cell_render.rendered_html {font-size:12pt;}
div.output {font-size:12pt; font-weight:bold}
div.input {font-family:Consolas ; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper {padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

<b><font size="6" color="red">ch.07 Attention</font></b>

# 1. 패키지 및 하이퍼 파라미터

In [2]:
import numpy as np
import pandas as pd
from time import time
from tensorflow.keras.layers import Input, LSTM, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical

# 하이터파라미터
MY_HIDDEN = 128
MY_EPOCH = 500

# 2. 학습데이터

In [3]:
raw = pd.read_csv('data/translate.csv', header=None)
eng_kor = raw.values.tolist() # 데이터프레임을 list로 변환
print(eng_kor[:3])
print('영_한 번역 데이터 개수', len(eng_kor))

[['cold', '감기'], ['come', '오다'], ['cook', '요리']]
영_한 번역 데이터 개수 110


# 3. 영어알파벳과 한글 문자 리스트 만들기

In [4]:
e_alpha = [c for c in 'SEPabcdefghijklmnopqrstuvwxyz']
korean = ''.join([data[1] for data in eng_kor])
k_ch = list(set([k for k in korean]))
k_ch.sort()

k_alpha = pd.read_csv('data/korean.csv', header=None)[0].tolist()
k_alpha == k_ch # 순서와 내용이 같은지 여부

True

In [5]:
from collections import Counter
list1 = ['가', '나', '다']
list2 = ['다', '가', '나']
print(list1 == list2)
print(Counter(list1) == Counter(list2))

False
True


In [6]:
alpha = e_alpha + k_alpha
print('영어와 한글 알파벳 :', alpha)
alpha_total_size = len(alpha) # 171개 (원핫인코딩사이즈)
print('전체 알파벳 갯수(원핫인코딩 사이즈) :', alpha_total_size)

영어와 한글 알파벳 : ['S', 'E', 'P', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '가', '각', '간', '감', '개', '거', '것', '게', '계', '고', '관', '광', '구', '굴', '규', '그', '금', '기', '깊', '나', '날', '남', '내', '넓', '녀', '노', '놀', '농', '높', '뉴', '늦', '다', '단', '도', '동', '들', '람', '랑', '래', '램', '류', '름', '릎', '리', '많', '망', '매', '머', '먼', '멍', '메', '명', '모', '목', '무', '물', '미', '바', '반', '방', '번', '복', '부', '분', '붕', '비', '뿌', '사', '상', '색', '생', '서', '선', '소', '손', '수', '쉽', '스', '시', '식', '실', '싸', '아', '약', '얇', '어', '언', '얼', '여', '연', '오', '옥', '왼', '요', '용', '우', '운', '움', '위', '유', '은', '을', '음', '의', '이', '익', '인', '읽', '입', '자', '작', '장', '적', '제', '좋', '주', '지', '짜', '쪽', '찾', '책', '출', '칙', '크', '키', '탈', '택', '통', '파', '팔', '편', '피', '핑', '한', '합', '해', '행', '험', '회', '획', '휴', '흐']
전체 알파벳 갯수(원핫인코딩 사이즈) : 171


# 4. 문자당 num를 갖는 dict

In [7]:
char_to_num = {c:i for i, c in enumerate(alpha)}
print(char_to_num)

{'S': 0, 'E': 1, 'P': 2, 'a': 3, 'b': 4, 'c': 5, 'd': 6, 'e': 7, 'f': 8, 'g': 9, 'h': 10, 'i': 11, 'j': 12, 'k': 13, 'l': 14, 'm': 15, 'n': 16, 'o': 17, 'p': 18, 'q': 19, 'r': 20, 's': 21, 't': 22, 'u': 23, 'v': 24, 'w': 25, 'x': 26, 'y': 27, 'z': 28, '가': 29, '각': 30, '간': 31, '감': 32, '개': 33, '거': 34, '것': 35, '게': 36, '계': 37, '고': 38, '관': 39, '광': 40, '구': 41, '굴': 42, '규': 43, '그': 44, '금': 45, '기': 46, '깊': 47, '나': 48, '날': 49, '남': 50, '내': 51, '넓': 52, '녀': 53, '노': 54, '놀': 55, '농': 56, '높': 57, '뉴': 58, '늦': 59, '다': 60, '단': 61, '도': 62, '동': 63, '들': 64, '람': 65, '랑': 66, '래': 67, '램': 68, '류': 69, '름': 70, '릎': 71, '리': 72, '많': 73, '망': 74, '매': 75, '머': 76, '먼': 77, '멍': 78, '메': 79, '명': 80, '모': 81, '목': 82, '무': 83, '물': 84, '미': 85, '바': 86, '반': 87, '방': 88, '번': 89, '복': 90, '부': 91, '분': 92, '붕': 93, '비': 94, '뿌': 95, '사': 96, '상': 97, '색': 98, '생': 99, '서': 100, '선': 101, '소': 102, '손': 103, '수': 104, '쉽': 105, '스': 106, '시': 107, '식': 108, '실': 109, '싸': 110,

In [73]:
# 숫자 -> 문자 
print('수->문 :', alpha[5])
print('문->수 :', char_to_num['c'])

수->문 : c
문->수 : 5


In [74]:
data = eng_kor[0]
print(data)
# char_to_num['c'], char_to_num['o'], char_to_num['l'], char_to_num['d']
print('인코더 입력 :', [char_to_num[ch] for ch in data[0]])
print('디코더 입력 :', [char_to_num[ch] for ch in 'S' + data[1]])
print('디코더 출력 :', [char_to_num[ch] for ch in  data[1] + 'E'])

['cold', '감기']
인코더 입력 : [5, 17, 14, 6]
디코더 입력 : [0, 32, 46]
디코더 출력 : [32, 46, 1]


In [44]:
# 원핫인코딩 방법1 : pd.get_dummies([5,3,7]) - 이 코드에서는 적용 불가
pd.get_dummies([5,3,7])

,3,5,7
0,0,1,0
1,1,0,0
2,0,0,1


In [45]:
# 원핫인코딩 방법2 : to_categorical([5,3,7], num_classes=10)
to_categorical([5,3,7], num_classes=10)

array([[0., 0., 0., 0., 0., 1., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 1., 0., 0.]], dtype=float32)

In [48]:
# 원핫인코딩 방법3: np.eye(10) [단위행렬]
np.eye(10)[[5,3,7]]

array([[0., 0., 0., 0., 0., 1., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 1., 0., 0.]])

# 5. 인코더 입력, 디코더 입력, 디코더 출력

- 인코더 입력(원핫인코딩O), 디코더 입력(원핫인코딩O) 디코더출력(원핫인코딩X)

In [8]:
def encoding(eng_kor = eng_kor):
    '인코더 입력, 디코더 입력, 디코더 출력 데이터를 return'
    enc_in = [] # 인코더 입력
    dec_in = [] # 디코더 입력
    dec_out = [] # 디코더 출력
    for data in eng_kor:
        # 인코더 입력 데이터(영어 -> 숫자 -> 원핫인코딩)
        eng = [char_to_num[ch] for ch in data[0]]
        #eng_one = to_categorical(eng, num_classes=alpha_total_size)
        eng_one = np.eye(alpha_total_size)[eng]
        enc_in.append(eng_one)
        
        # 디코더 입력 데이터('S한글' -> 숫자 -> 원핫인코딩)
        kor = [char_to_num[ch] for ch in 'S' + data[1]]
        kor_one = np.eye(alpha_total_size)[kor]
        dec_in.append(kor_one)
        
        
        # 디코더 출력('한글E' -> 숫자)
        kor = [char_to_num[ch] for ch in data[1]+'E']
        dec_out.append(kor)
        
    # 인공신경망에 넣을 데이터이므로 numpy 배열로 전환
    enc_in = np.array(enc_in)
    dec_in = np.array(dec_in)
    dec_out = np.array(dec_out)
    return enc_in, dec_in, dec_out
        
sample = [['wood', '나무'], ['word', '단어']]

In [76]:
# RNN 분류분석 : 타겟변수를 원핫인코딩 (시스템에거 원핫인코딩을 의뢰)
X_enc, X_dec, y_dec = encoding(sample)
X_enc.shape, X_dec.shape, y_dec.shape

((2, 4, 171), (2, 3, 171), (2, 3))

## 축 증가

In [62]:
y_dec.reshape(2,3,1) # 방법1

array([[[ 48],
        [ 83],
        [  1]],

       [[ 61],
        [114],
        [  1]]])

In [67]:
np.expand_dims(y_dec, axis=-1) # 방법2

array([[[ 48],
        [ 83],
        [  1]],

       [[ 61],
        [114],
        [  1]]])

In [64]:
y_dec[..., np.newaxis] # 방법3

array([[[ 48],
        [ 83],
        [  1]],

       [[ 61],
        [114],
        [  1]]])

In [66]:
y_dec[:,:,None] # 방법4

array([[[ 48],
        [ 83],
        [  1]],

       [[ 61],
        [114],
        [  1]]])

# 6. 전체 번역 데이터(독립변수, 타겟변수)

In [9]:
X_enc, X_dec, y_dec = encoding(eng_kor)
Y_dec = np.expand_dims(y_dec,axis=-1)
X_enc.shape, X_dec.shape, Y_dec.shape

((110, 4, 171), (110, 3, 171), (110, 3, 1))

# 7_1. Seq2Seq모델구현

In [81]:
# 인코더 LSTM 구현
ENC_IN = Input(shape=(4, alpha_total_size))
# 윗출력, state_c, state_h
_, state_c, state_h = LSTM(
                    units=MY_HIDDEN,
                    # return_sequences=False, 윗출력 받지 않음(기본값)
                    return_state= True, # 우출력 받음
                )(ENC_IN)
# 인코더와 디코더를 연결할  link
link = [state_c, state_h]

# 디코더 구현 : return_sequences = True 위로 올라가는 출력값 사용
DEC_IN = Input(shape=(3, alpha_total_size))
DEC_MID = LSTM(units=MY_HIDDEN,
               return_sequences= True)(DEC_IN,
                                      initial_state=link)

# 최종 출력층
DEC_OUT = Dense(units=alpha_total_size,
               activation='softmax')(DEC_MID)

# 모델
model = Model(inputs = [ENC_IN, DEC_IN],
              outputs = DEC_OUT)
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_3 (InputLayer)           [(None, 4, 171)]     0           []                               
                                                                                                  
 input_4 (InputLayer)           [(None, 3, 171)]     0           []                               
                                                                                                  
 lstm_2 (LSTM)                  [(None, 128),        153600      ['input_3[0][0]']                
                                 (None, 128),                                                     
                                 (None, 128)]                                                     
                                                                                              

# 7_2 Attention 모델 구현

In [10]:
from tensorflow.keras.layers import Attention, Concatenate

# 인코더 LSTM 구현
ENC_IN = Input(shape=(4, alpha_total_size))
# 윗출력, state_c, state_h
ENC_OUT, state_c, state_h = LSTM(
                    units=MY_HIDDEN,
                    return_sequences=True,
                    return_state= True, # 우출력 받음
                )(ENC_IN)
# 인코더와 디코더를 연결할  link
link = [state_c, state_h]

# 디코더 구현 : return_sequences = True 위로 올라가는 출력값 사용
DEC_IN = Input(shape=(3, alpha_total_size))
DEC_MID, _, _ = LSTM(units=MY_HIDDEN,
               return_state=True,
               return_sequences= True)(DEC_IN,
                                      initial_state=link)
# 어텐션 메커니즘
CONTEXT_VECTOR = Attention()([DEC_MID,ENC_OUT])

# 컨텍스트벡터와 디코더(LSTM 출력을 결합)
CONTEXT_AND_LSTM_OUT = Concatenate()([CONTEXT_VECTOR, DEC_MID])

# 최종 출력층
OUT = Dense(units=alpha_total_size,
               activation='softmax')(CONTEXT_AND_LSTM_OUT)

# 모델
model = Model(inputs = [ENC_IN, DEC_IN],
              outputs = OUT)
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 4, 171)]     0           []                               
                                                                                                  
 input_2 (InputLayer)           [(None, 3, 171)]     0           []                               
                                                                                                  
 lstm (LSTM)                    [(None, 4, 128),     153600      ['input_1[0][0]']                
                                 (None, 128),                                                     
                                 (None, 128)]                                                     
                                                                                              

# 8. 학습과정 설정 및 학습하기

In [11]:
model.compile(loss='sparse_categorical_crossentropy',
              optimizer='rmsprop',
              metrics=['acc'])

start = time()
hist = model.fit([X_enc, X_dec], Y_dec,
                epochs=MY_EPOCH,
                verbose=1)
end = time()

print('학습시간 :', end-start)

Epoch 1/500
4/4 [==============================] - 2s 7ms/step - loss: 5.1164 - acc: 0.2273
Epoch 2/500
4/4 [==============================] - 0s 7ms/step - loss: 4.9728 - acc: 0.3333
Epoch 3/500
4/4 [==============================] - 0s 7ms/step - loss: 4.3275 - acc: 0.3333
Epoch 4/500
4/4 [==============================] - 0s 9ms/step - loss: 3.4650 - acc: 0.3333
Epoch 5/500
4/4 [==============================] - 0s 8ms/step - loss: 3.4057 - acc: 0.3333
Epoch 6/500
4/4 [==============================] - 0s 6ms/step - loss: 3.3623 - acc: 0.3333
Epoch 7/500
4/4 [==============================] - 0s 6ms/step - loss: 3.3313 - acc: 0.3333
Epoch 8/500
4/4 [==============================] - 0s 7ms/step - loss: 3.3011 - acc: 0.3333
Epoch 9/500
4/4 [==============================] - 0s 8ms/step - loss: 3.2778 - acc: 0.3333
Epoch 10/500
4/4 [==============================] - 0s 6ms/step - loss: 3.2513 - acc: 0.3333
Epoch 11/500
4/4 [==============================] - 0s 6ms/step - loss: 3.2331 

4/4 [==============================] - 0s 6ms/step - loss: 0.4277 - acc: 0.9636
Epoch 90/500
4/4 [==============================] - 0s 6ms/step - loss: 0.4123 - acc: 0.9727
Epoch 91/500
4/4 [==============================] - 0s 11ms/step - loss: 0.3960 - acc: 0.9758
Epoch 92/500
4/4 [==============================] - 0s 12ms/step - loss: 0.3760 - acc: 0.9727
Epoch 93/500
4/4 [==============================] - 0s 5ms/step - loss: 0.3694 - acc: 0.9727
Epoch 94/500
4/4 [==============================] - 0s 5ms/step - loss: 0.3402 - acc: 0.9788
Epoch 95/500
4/4 [==============================] - 0s 6ms/step - loss: 0.3242 - acc: 0.9758
Epoch 96/500
4/4 [==============================] - 0s 6ms/step - loss: 0.3095 - acc: 0.9848
Epoch 97/500
4/4 [==============================] - 0s 7ms/step - loss: 0.2979 - acc: 0.9909
Epoch 98/500
4/4 [==============================] - 0s 6ms/step - loss: 0.2825 - acc: 0.9879
Epoch 99/500
4/4 [==============================] - 0s 8ms/step - loss: 0.2709 - 

4/4 [==============================] - 0s 6ms/step - loss: 0.0047 - acc: 1.0000
Epoch 177/500
4/4 [==============================] - 0s 8ms/step - loss: 0.0048 - acc: 1.0000
Epoch 178/500
4/4 [==============================] - 0s 7ms/step - loss: 0.0043 - acc: 1.0000
Epoch 179/500
4/4 [==============================] - 0s 7ms/step - loss: 0.0038 - acc: 1.0000
Epoch 180/500
4/4 [==============================] - 0s 9ms/step - loss: 0.0037 - acc: 1.0000
Epoch 181/500
4/4 [==============================] - 0s 6ms/step - loss: 0.0050 - acc: 1.0000
Epoch 182/500
4/4 [==============================] - 0s 6ms/step - loss: 0.0081 - acc: 0.9970
Epoch 183/500
4/4 [==============================] - 0s 7ms/step - loss: 0.0067 - acc: 1.0000
Epoch 184/500
4/4 [==============================] - 0s 7ms/step - loss: 0.0039 - acc: 1.0000
Epoch 185/500
4/4 [==============================] - 0s 6ms/step - loss: 0.0027 - acc: 1.0000
Epoch 186/500
4/4 [==============================] - 0s 6ms/step - loss: 0

4/4 [==============================] - 0s 9ms/step - loss: 3.2957e-05 - acc: 1.0000
Epoch 262/500
4/4 [==============================] - 0s 6ms/step - loss: 3.0823e-05 - acc: 1.0000
Epoch 263/500
4/4 [==============================] - 0s 6ms/step - loss: 2.7825e-05 - acc: 1.0000
Epoch 264/500
4/4 [==============================] - 0s 7ms/step - loss: 2.7007e-05 - acc: 1.0000
Epoch 265/500
4/4 [==============================] - 0s 9ms/step - loss: 2.4941e-05 - acc: 1.0000
Epoch 266/500
4/4 [==============================] - 0s 8ms/step - loss: 2.4579e-05 - acc: 1.0000
Epoch 267/500
4/4 [==============================] - 0s 6ms/step - loss: 2.2497e-05 - acc: 1.0000
Epoch 268/500
4/4 [==============================] - 0s 7ms/step - loss: 2.0727e-05 - acc: 1.0000
Epoch 269/500
4/4 [==============================] - 0s 7ms/step - loss: 1.9861e-05 - acc: 1.0000
Epoch 270/500
4/4 [==============================] - 0s 9ms/step - loss: 1.8633e-05 - acc: 1.0000
Epoch 271/500
4/4 [===============

4/4 [==============================] - 0s 6ms/step - loss: 1.9214e-06 - acc: 1.0000
Epoch 345/500
4/4 [==============================] - 0s 7ms/step - loss: 1.8694e-06 - acc: 1.0000
Epoch 346/500
4/4 [==============================] - 0s 6ms/step - loss: 1.8337e-06 - acc: 1.0000
Epoch 347/500
4/4 [==============================] - 0s 6ms/step - loss: 1.7896e-06 - acc: 1.0000
Epoch 348/500
4/4 [==============================] - 0s 6ms/step - loss: 1.7610e-06 - acc: 1.0000
Epoch 349/500
4/4 [==============================] - 0s 6ms/step - loss: 1.7213e-06 - acc: 1.0000
Epoch 350/500
4/4 [==============================] - 0s 6ms/step - loss: 1.6863e-06 - acc: 1.0000
Epoch 351/500
4/4 [==============================] - 0s 5ms/step - loss: 1.6646e-06 - acc: 1.0000
Epoch 352/500
4/4 [==============================] - 0s 6ms/step - loss: 1.6198e-06 - acc: 1.0000
Epoch 353/500
4/4 [==============================] - 0s 5ms/step - loss: 1.5891e-06 - acc: 1.0000
Epoch 354/500
4/4 [===============

4/4 [==============================] - 0s 6ms/step - loss: 5.8124e-07 - acc: 1.0000
Epoch 428/500
4/4 [==============================] - 0s 6ms/step - loss: 5.8340e-07 - acc: 1.0000
Epoch 429/500
4/4 [==============================] - 0s 6ms/step - loss: 5.7979e-07 - acc: 1.0000
Epoch 430/500
4/4 [==============================] - 0s 6ms/step - loss: 5.7220e-07 - acc: 1.0000
Epoch 431/500
4/4 [==============================] - 0s 5ms/step - loss: 5.5956e-07 - acc: 1.0000
Epoch 432/500
4/4 [==============================] - 0s 11ms/step - loss: 5.5739e-07 - acc: 1.0000
Epoch 433/500
4/4 [==============================] - 0s 11ms/step - loss: 5.5848e-07 - acc: 1.0000
Epoch 434/500
4/4 [==============================] - 0s 5ms/step - loss: 5.5378e-07 - acc: 1.0000
Epoch 435/500
4/4 [==============================] - 0s 6ms/step - loss: 5.5342e-07 - acc: 1.0000
Epoch 436/500
4/4 [==============================] - 0s 7ms/step - loss: 5.4728e-07 - acc: 1.0000
Epoch 437/500
4/4 [=============

In [84]:
model.save('data/seq2seq.h5')

## ※ 12월 5일

In [14]:
from tensorflow.keras.models import load_model
model = load_model('data/seq2seq.h5')

In [12]:
# 쉬운문제
easy_test = [['find','PP'],
            ['cold', 'PP'],
            ['cook', 'PP'],
            ['date', 'PP'],
            ['desk', 'PP']]
enc_in, dec_in, dec_out,= encoding(easy_test)
enc_in.shape, dec_in.shape

((5, 4, 171), (5, 3, 171))

In [13]:
pred = model.predict([enc_in, dec_in])
pred.shape

1/1 [==============================] - 0s 493ms/step


(5, 3, 171)

In [14]:
# pred[0]
for i in range(len(pred)):
    eng=easy_test[i]
    hat = pred[i].argmax(axis=-1)[:-1]
    # kor = alpha[[hat[0]]] + alpha[[hat[1]]]
    kor = ''.join([alpha[num] for num in hat])
    print("{} => {}{}".format(eng, kor, hat))

['find', 'PP'] => 찾다[148  60]
['cold', 'PP'] => 감기[32 46]
['cook', 'PP'] => 요리[122  72]
['date', 'PP'] => 날짜[ 49 146]
['desk', 'PP'] => 책상[149  97]


In [15]:
# 어려운 문제
hard_test = [['love','PP'],
            ['lvoe','PP'],
            ['loev','PP'],
            ['olve','PP'],
            ['evol','PP'],]

enc_in, dec_in, _ = encoding(hard_test)
enc_in.shape, dec_in.shape

((5, 4, 171), (5, 3, 171))

In [16]:
pred = model.predict([enc_in, dec_in], verbose=0).argmax(axis=-1)
pred

array([[96, 66,  1],
       [96, 66,  1],
       [96, 66,  1],
       [96, 66,  1],
       [75, 96,  1]], dtype=int64)

In [17]:
for i in range(len(pred)):
    eng = hard_test[i][0]
    kor = [alpha[num] for num in pred[i]]
    print('{}=>{}{}'.format(eng, kor[:-1], pred[i][:-1]))

love=>['사', '랑'][96 66]
lvoe=>['사', '랑'][96 66]
loev=>['사', '랑'][96 66]
olve=>['사', '랑'][96 66]
evol=>['매', '사'][75 96]
